# Reproducing Martel et al. (2025) from dandiset 001174: activity, task responses and coactivation

Paper. Martel A-C, Pittard D, Devergnas A, Risk B, Nassi JJ, Yu W, Downer JD, Wichmann T, Galvan A
(2025). Microendoscopic calcium imaging in motor cortices of macaques during rest and movement.
*iScience* 28, 112767. [doi:10.1016/j.isci.2025.112767](https://doi.org/10.1016/j.isci.2025.112767).
The claims file (`claims/001174/10.1016-j.isci.2025.112767.yaml`) was written from the version of
record as served by Europe PMC (PMC12192352), with the STAR Methods and figure legends; the
supplement (Figures S1 to S3, Tables S1 to S3) was not in the text read.

Dandiset. [DANDI:001174](https://dandiarchive.org/dandiset/001174/0.250331.2218), version
0.250331.2218. This is the version the paper's key resources table cites
(doi:10.48324/dandi.001174/0.250331.2218) and the only published version; the data check compares
it with the draft.

What the dandiset holds. 45 NWB files, one per session and condition: the miniscope movie, the
footprints of the accepted cells, one trace per cell (the CNMFe denoised trace, see the data check)
and the deconvolved events as a frame by cell array of amplitudes. It holds no TTL or trial data and
no raw (undenoised) traces.

Outside inputs. The paper's per-cell descriptors table (Zenodo
[doi:10.5281/zenodo.14727695](https://doi.org/10.5281/zenodo.14727695)) supplies the cells' first
occurrences, which the dandiset does not mark; the link claim tests it. The paper's Jaccard output
table (Zenodo [doi:10.5281/zenodo.15116617](https://doi.org/10.5281/zenodo.15116617)) is shown
next to this attempt's values, and decides nothing. The paper's table of z-scored trial windows (Zenodo
[doi:10.5281/zenodo.14976651](https://doi.org/10.5281/zenodo.14976651)) is used for a check of the
Figure 2 classification on the paper's own processed data, which decides nothing. Method details come from the paper's code
(Zenodo doi:10.5281/zenodo.14963213 and doi:10.5281/zenodo.14977106), as the claims file cites
it. The sequence claims (Figure 4) are in the second notebook, `sequences_martel_2025.ipynb`.

| Claim | Tested here |
|---|---|
| zenodo-descriptors-link | tested |
| cell-counts | tested |
| cells-tracked-18pct | blocked: metadata insufficient |
| fig1d-rate-no-change | tested |
| fig1e-ieicv-no-change | tested |
| fig1f-amplitude-u-sma-spont-higher | tested |
| fig1d-rate-m1-above-sma-u | tested |
| fig1f-amplitude-m1-above-sma-u-task | tested |
| fig2-q-sma-modulated-34pct, fig2c-q-sma-direction, fig2-q-magnitude-no-difference | blocked: data missing (no TTL, no raw traces) |
| figs1-u-sma-direction, figs1-u-m1-target-onset, figs1-u-m1-movement-onset | blocked: data missing |
| fig3-synchronized-pairs-exist | tested |
| fig3ab-synchrony-not-distance | tested |
| fig3c-sync-no-condition-change | tested |
| fig4a, fig4b, fig4d, fig4c (sequences) | in `sequences_martel_2025.ipynb` |
| histology-lens-placement, histology-gcamp-not-gabaergic | blocked: other (an anatomist's reading of slide scans) |

In [ ]:
import json, urllib.request
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import sma_m1 as h

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("dandiset", h.DANDISET, "version", h.VERSION)
files = h.load_all()
ses = h.sessions(files)
print(len(files), "files loaded")

## Data check

What was loaded, per file: subject, site (from `session_description`), date, condition, frames,
sampling rate, duration and cells. Then the checks of the description against the values.

In [ ]:
inv = pd.DataFrame([{
    "path": x["path"], "subject": x["subject"], "region": x["region"], "date": x["date"],
    "condition": x["condition"], "frames": x["events"].shape[0], "rate_Hz": round(x["rate_ea"], 4),
    "imaging_rate_Hz": x["imaging_rate"], "minutes": round((x["events"].shape[0] - 1) / x["rate_ea"] / 60, 2),
    "cells": x["events"].shape[1], "movie": "x".join(map(str, x["movie_shape"][1:])),
    "mask": "x".join(map(str, x["mask_shape"][1:])), "events": int((x["events"] > 0).sum()),
} for x in files])
print(inv.drop(columns="path").to_string())
print()
print(inv.groupby(["subject", "region", "condition"]).agg(files=("cells", "size"), cells=("cells", "sum"),
      minutes_min=("minutes", "min"), minutes_max=("minutes", "max")).to_string())
print()
print("subjects:", {x["subject"]: (x["sex"], x["age"]) for x in files})
print("indicator field:", sorted({x["indicator"] for x in files}), " location field:", sorted({x["location"] for x in files}))
print("file names without a session label:", [x["path"] for x in files if "ses-" not in x["path"]])

In [ ]:
# Draft against the pinned version: same assets?
def asset_list(v):
    url = f"{h.API}/dandisets/{h.DANDISET}/versions/{v}/assets/?page_size=500"
    with urllib.request.urlopen(url) as r:
        return {a["path"]: (a["size"], a["blob"] if "blob" in a else None) for a in json.load(r)["results"]}
pub, draft = asset_list(h.VERSION), asset_list("draft")
print("published assets", len(pub), " draft assets", len(draft))
print("same paths and sizes in draft:", {k: v[0] for k, v in pub.items()} == {k: v[0] for k, v in draft.items()})

In [ ]:
# Pairs of files of a session: the two conditions share the footprints (same cells), and no
# recording is deposited twice (identical event arrays across files).
rows = []
for key, d in sorted(ses.items()):
    sp, ta = d.get("spontaneous"), d.get("task")
    rows.append({"subject": key[0], "region": key[1], "date": key[2], "spont": sp is not None, "task": ta is not None,
                 "cells": (sp or ta)["events"].shape[1],
                 "same_footprints": (sp["mask_hash"] == ta["mask_hash"]) if (sp and ta) else None})
pairs = pd.DataFrame(rows)
print(pairs.to_string())
hashes = {}
for x in files:
    hashes.setdefault(hash(x["events"].tobytes()), []).append(x["path"])
print("files with identical event arrays:", [v for v in hashes.values() if len(v) > 1])
fp = {}
for x in files:
    fp.setdefault((x["subject"], x["region"], x["mask_hash"]), []).append(x["date"])
print("footprint sets shared across dates:", {k[:2]: v for k, v in fp.items() if len(set(v)) > 1})

In [ ]:
# Which trace is in Fluorescence/RoiResponseSeries? A raw trace is noisy; the CNMFe denoised trace
# decays by one constant factor per frame between events (an AR(1) model). Ratios of successive
# samples, per file: the share of ratios equal (to 1e-4) to the cell's modal decay factor.
def decay_share(x):
    d = x["denoised"]
    out = []
    for c in range(d.shape[1]):
        v = d[:, c]
        r = v[1:] / np.where(v[:-1] > 0, v[:-1], np.nan)
        r = r[np.isfinite(r)]
        if len(r) < 10:
            continue
        mode = np.median(r[r < 1])
        out.append(np.mean(np.abs(r - mode) < 1e-4))
    return np.median(out)
sh = np.array([decay_share(x) for x in files])
print("median share of samples that decay by the cell's constant factor, per file: min %.3f, median %.3f" % (sh.min(), np.median(sh)))
x = files[0]
print("example", x["path"], "cell 0, first 12 samples:", np.round(x["denoised"][:12, 0], 3))
print("NaN samples in the traces: %d files have some, at most %.2f%% of samples" %
      (sum(np.isnan(x["denoised"]).any() for x in files), 100 * max(np.isnan(x["denoised"]).mean() for x in files)))
ev_amp = np.concatenate([x["events"][x["events"] > 0] for x in files])
print("event amplitudes: n=%d, range %.2f to %.2f, median %.2f (unit 'fluorescence')" % (len(ev_amp), ev_amp.min(), ev_amp.max(), np.median(ev_amp)))
cen = np.concatenate([x["centroid"] for x in files])
print("centroids (pixels): x %.1f to %.1f, y %.1f to %.1f" % (cen[:, 0].min(), cen[:, 0].max(), cen[:, 1].min(), cen[:, 1].max()))

In [ ]:
# Shared clock of events and traces. Both are per frame in the same file; an event should fall where
# the denoised trace jumps up. Contrast: the share of event frames where the trace rises by more than
# 5% against the same with the events shifted by k frames.
def rise_share(x, k):
    d, e = x["denoised"], x["events"] > 0
    up = np.zeros_like(e)
    up[1:] = (d[1:] > 1.05 * d[:-1]) & np.isfinite(d[1:]) & np.isfinite(d[:-1])
    es = np.roll(e, k, axis=0)
    return up[es].mean()
lag = pd.DataFrame({k: [rise_share(x, k) for x in files] for k in (-2, -1, 0, 1, 2, 10)})
print("share of event frames on a rise of the denoised trace, by shift of the events (frames):")
print(lag.describe().loc[["min", "50%", "max"]].round(3).to_string())
print("files where the unshifted share is not the largest:", int((lag.idxmax(axis=1) != 0).sum()))

In [ ]:
# What the NWB files hold besides the imaging: no trial table, no TTL or behavioural channel, and no
# cross-session cell identity. Every one of the 45 files is opened (groups and table columns only),
# and the archive's per-asset metadata (variableMeasured) is listed.
import h5py, remfile
from concurrent.futures import ThreadPoolExecutor
def contents(row):
    with h5py.File(remfile.File(h._s3_url(row["asset_id"])), "r") as f:
        ps = f["processing/ophys/ImageSegmentation/PlaneSegmentation"]
        return {"path": row["path"],
                "acquisition": ",".join(sorted(f["acquisition"].keys())),
                "processing": ",".join(sorted(f["processing"].keys())),
                "ophys": ",".join(sorted(f["processing/ophys"].keys())),
                "stimulus": ",".join(sorted(k for g in ("stimulus/presentation", "stimulus/templates") if g in f for k in f[g].keys())),
                "analysis": ",".join(sorted(f["analysis"].keys())) if "analysis" in f else "",
                "intervals": "intervals" in f,
                "plane_segmentation_columns": ",".join(sorted(k for k in ps.keys())),
                "colnames": ps.attrs.get("colnames").tolist() if "colnames" in ps.attrs else None}
assets = h.list_assets()
with ThreadPoolExecutor(12) as ex:
    CT = pd.DataFrame(list(ex.map(contents, [r for _, r in assets.iterrows()])))
for col in ["acquisition", "processing", "ophys", "stimulus", "analysis", "intervals", "plane_segmentation_columns"]:
    print(col, "->", CT[col].value_counts().to_dict())
print("PlaneSegmentation colnames:", {str(v) for v in CT.colnames})
print("files whose PlaneSegmentation ids are 0..n-1:", sum(bool((x["roi_ids"] == np.arange(len(x["roi_ids"]))).all()) for x in files), "of", len(files))
def var_measured(aid):
    with urllib.request.urlopen(f"{h.API}/dandisets/{h.DANDISET}/versions/{h.VERSION}/assets/{aid}/") as r:
        return tuple(sorted(v.get("value", str(v)) if isinstance(v, dict) else str(v) for v in json.load(r).get("variableMeasured", [])))
with ThreadPoolExecutor(12) as ex:
    vm = list(ex.map(var_measured, assets.asset_id))
print("archive variableMeasured over the 45 assets:", pd.Series(vm).value_counts().to_dict())


Checks of the description against the data.

- The dandiset holds 45 files over the sessions the claims file lists: each session of Q, U and V has a
  spontaneous and an arm-reaching file, F has spontaneous files only, and the 2022-09-15
  spontaneous file of Q is named `sub-Q/sub-Q_ophys.nwb`, without a session label (counts printed above).
- The two files of a session share their footprints (column `same_footprints`), so their cells are the
  same cells in the same order. No two files hold identical event arrays.
- V's sessions have arm-reaching files, while the paper says only Q and U performed the task.
- The Methods give 10 Hz; the U SMA session of 2023-03-14 is sampled at 20 Hz (both files).
- The trace in `Fluorescence/RoiResponseSeries` decays by one constant factor per frame between
  events: it is the denoised CNMFe trace, though the files call it "Fluorescence data". The raw traces
  that Figure 2 uses are not in the dandiset.
- Event frames fall on rises of the denoised trace, and much less often when the events are shifted,
  which is consistent with a shared clock of events and traces.
- `indicator` and `location` are "N/A" in every file; the site is only in `session_description`.
- None of the 45 files has a trial table, an intervals group, a stimulus or a TTL or behavioural
  channel, and the archive's metadata lists only imaging variables for every asset, so the task events
  of Figure 2 and Figure S1 cannot be located in the recordings.
- The ids of `PlaneSegmentation` are 0..n-1 in every file and it has no other column that could name a
  cell across sessions, so the dandiset carries no cross-session identity.

## zenodo-descriptors-link

> The per-cell descriptors computed from the dandiset (event rate, IEI CV, mean amplitude, spontaneous
> and task) agree with the per-cell table the paper deposited on Zenodo, so that table can give the
> cells' first occurrences.

Cells are matched by subject, site, date and the index of the accepted cell (the table's `cells`
column). The match of identity is checked with the centroids, which the table also gives; the
criterion judges the descriptors.

In [ ]:
z, z_md5 = h.zenodo_descriptors()
print("Zenodo table:", z_md5, len(z), "rows,", z["Global_cell_ID"].nunique(), "global cell ids")
D = h.descriptor_table(files)
m = z.merge(D, on=["subject", "region", "date", "condition", "cell"], how="left", indicator=True)
print("table rows matched to a dandiset cell:", (m["_merge"] == "both").sum(), "of", len(m))
print("unmatched table rows (no such file):")
print(m[m["_merge"] == "left_only"].groupby(["subject", "region", "condition"]).size().to_string())
mm = m[m["_merge"] == "both"].copy()
mm["d_x"] = mm["X_centroid"] - mm["cx"]
mm["d_y"] = mm["Y_centroid"] - mm["cy"]
print("centroid differences (pixels), table minus dandiset: x median %.2f, |x| 95th pct %.2f; y median %.2f, |y| 95th pct %.2f"
      % (mm.d_x.median(), mm.d_x.abs().quantile(.95), mm.d_y.median(), mm.d_y.abs().quantile(.95)))
print("cells with both centroid differences under 3 pixels: %.3f" % ((mm.d_x.abs() < 3) & (mm.d_y.abs() < 3)).mean())

In [ ]:
def rel(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.abs(a - b) / np.abs(a)
    r[(a == 0) & (b == 0)] = 0
    return r
mm["rate_rel"] = rel(mm["Event Rate"], mm["rate"])
mm["amp_rel"] = rel(mm["Mean_Amplitude"], mm["amplitude"])
mm["n_table"] = mm["Event Rate"] * mm["duration_s"]
both = mm[mm["Mean_Amplitude"].notna() & mm["amplitude"].notna()]
ok = (mm["rate_rel"] < 0.01) & ((mm["amp_rel"] < 0.01) | (mm["Mean_Amplitude"].isna() & mm["amplitude"].isna()))
cells_ok = mm.assign(ok=ok).groupby(["subject", "region", "date", "cell"])["ok"].all()
print("cell-conditions with rate within 1%%: %.3f; amplitude within 1%% (both defined, n=%d): %.3f"
      % ((mm.rate_rel < 0.01).mean(), len(both), (both.amp_rel < 0.01).mean()))
print("matched cells with rate and amplitude within 1%% in every condition: %d of %d = %.3f"
      % (cells_ok.sum(), len(cells_ok), cells_ok.mean()))
print("table rate / dandiset rate: median %.3f, 5th to 95th pct %.3f to %.3f"
      % ((mm["Event Rate"] / mm["rate"]).replace(np.inf, np.nan).median(),
         *(mm["Event Rate"] / mm["rate"]).replace(np.inf, np.nan).quantile([.05, .95])))
print("table CV / dandiset CV: median %.3f" % (mm["CV"] / mm["iei_cv"]).median())
per = mm.assign(ok=ok, ratio=mm["Event Rate"] / mm["rate"]).groupby(["subject", "region", "date", "condition"]).agg(
    cells=("ok", "size"), within_1pct=("ok", "mean"), median_rate_ratio=("ratio", "median"))
print(per.round(3).to_string())

In [ ]:
# Where the event counts differ: one-frame inter-event intervals. The table's median IEI is one frame
# for many cells; compare the share of cells whose median IEI is one frame (0.1 s) in each source.
mm["med_isi_dandi"] = np.nan
for (s, r, d, c), g in mm.groupby(["subject", "region", "date", "condition"]):
    x = ses[(s, r, d)][c]
    for i, row in g.iterrows():
        idx = np.flatnonzero(x["events"][:, row["cell"]] > 0)
        if len(idx) > 1:
            mm.loc[i, "med_isi_dandi"] = np.median(np.diff(idx)) / x["rate_ea"]
one = lambda v: (np.abs(v - 0.1) < 0.02)
sel = mm["Median_ISI"].notna() & mm["med_isi_dandi"].notna()
print("cell-conditions with a median IEI of one frame: table %.3f, dandiset %.3f (n=%d)"
      % (one(mm.loc[sel, "Median_ISI"]).mean(), one(mm.loc[sel, "med_isi_dandi"]).mean(), sel.sum()))

In [ ]:
# The two U SMA sessions where the table and the dandiset differ most (2023-03-07 and 2023-03-14, the
# 20 Hz session), cell by cell: the table's event count (its rate times the file's duration) against the
# dandiset's, and the mean amplitudes. A swap of conditions would make the table's spontaneous counts
# follow the dandiset's task counts; the rank correlation is given for the same and the swapped condition.
for d in ("2023-03-07", "2023-03-14", "2023-02-27"):
    g = mm[(mm.subject == "U") & (mm.region == "SMA") & (mm.date == d)]
    pv = g.pivot_table(index="cell", columns="condition", values=["n_table", "n_events", "Mean_Amplitude", "amplitude"])
    print(f"U SMA {d}: cells {len(pv)}; table events (rate x duration) and dandiset events per condition, summed:",
          {c: (round(pv["n_table"][c].sum()), int(pv["n_events"][c].sum())) for c in ("spontaneous", "task")})
    for c, cs in (("spontaneous", "spontaneous"), ("spontaneous", "task"), ("task", "task"), ("task", "spontaneous")):
        a, b = pv["n_table"][c], pv["n_events"][cs]
        print(f"   table {c:11s} against dandiset {cs:11s}: Spearman %.3f" % stats.spearmanr(a, b)[0])
    amp = g.dropna(subset=["Mean_Amplitude", "amplitude"])
    print("   amplitude, table against dandiset: median %.3f and %.3f, dandiset higher in %d of %d cell-conditions"
          % (amp.Mean_Amplitude.median(), amp.amplitude.median(), int((amp.amplitude > amp.Mean_Amplitude).sum()), len(amp)))
print(mm[(mm.subject == "U") & (mm.region == "SMA") & mm.date.isin(["2023-03-07", "2023-03-14"])][
    ["date", "condition", "cell", "Event Rate", "rate", "n_table", "n_events", "Mean_Amplitude", "amplitude"]].round(4).to_string())


**Result: not reproduced.** The table's cells are found in the dandiset: 565 of 626 table rows match a cell by session and index (the 61 unmatched are F task rows, and F has no task files), and 93.8% of the matched cells have both centroid coordinates within 3 pixels of the dandiset's footprint centroids, so the identity of the cells holds. The descriptors do not agree to the criterion's 1%: 86 of 313 matched cells (27.5%) have rate and amplitude within 1% in every condition, below the 50% the criterion asks. The table's rates are higher than the dandiset's by a median factor of 1.042 (5th to 95th percentile 0.999 to 1.499), and agree within 1% for 40.9% of cell-conditions; amplitudes agree within 1% for 66.3%. Two U SMA sessions differ by large factors (2023-03-07 spontaneous 19.999, 2023-03-14 task 0.042). One-frame inter-event intervals are rarer in the dandiset (median IEI of one frame in 3.9% of cell-conditions, against 11.9% in the table), consistent with events missing from the dandiset's event arrays. The two U SMA sessions that differ most are not a swap of conditions: in 2023-03-07 the table's task counts follow the dandiset's task counts (Spearman 0.975) while its spontaneous counts follow neither file (-0.354 and 0.205), and in 2023-03-14 each condition follows its own file best (0.829 and 0.786). In both, the dandiset has fewer spontaneous events (4 against 133 and 78 against 246, over the first-occurrence cells) and a higher amplitude in every cell-condition (medians 3.702 against 3.080 and 3.488 against 2.987), which looks like a different event detection in those files; the well-matched 2023-02-27 has amplitudes higher in 13 of 28. The table is still used below for the cells' first occurrences, which rest on the identity, not on the descriptors.

## cell-counts

> In SMA 129, 61 and 63 cells were identified in monkeys Q, F and U, and in M1 37 and 23 cells in
> monkeys U and V.

Reading (i): unique cells, each counted once, which needs the first occurrences of the Zenodo table
(through the link above). Reading (ii): the sum of accepted cells over the sessions the claims file
lists (F without 2024-03-19, which the Jaccard code leaves out with one cell; F with it is shown).

In [ ]:
expected = {("Q", "SMA"): 129, ("F", "SMA"): 61, ("U", "SMA"): 63, ("U", "M1"): 37, ("V", "M1"): 23}
first = mm[mm.condition == "spontaneous"]
uniq = first.groupby(["subject", "region"])["cell"].size()
uniq_no319 = first[~((first.subject == "F") & (first.date == "2024-03-19"))].groupby(["subject", "region"])["cell"].size()
summed = pd.Series({k[:2]: 0 for k in ses})
summed_no319 = summed.copy()
for k, d in ses.items():
    n = (d.get("spontaneous") or d.get("task"))["events"].shape[1]
    summed[k[:2]] += n
    if k != ("F", "SMA", "2024-03-19"):
        summed_no319[k[:2]] += n
tab = pd.DataFrame({"paper": pd.Series(expected), "unique (i)": uniq, "unique (i), F without 03-19": uniq_no319,
                    "sum over sessions (ii)": summed, "sum (ii), F without 03-19": summed_no319})
tab["tolerance"] = tab["paper"] * 0.1
for c in ["unique (i)", "unique (i), F without 03-19", "sum over sessions (ii)", "sum (ii), F without 03-19"]:
    tab["ok " + c] = (tab[c] - tab["paper"]).abs() <= tab["tolerance"]
print(tab.to_string())
print("all five within tolerance: reading (i)", tab["ok unique (i), F without 03-19"].all(), "| reading (ii)", tab["ok sum (ii), F without 03-19"].all())
print("unique cells in total:", int(uniq.sum()), "| accepted cells summed over sessions:", int(summed.sum()))

**Result: reproduced.** Under reading (i), unique cells counted once, the counts are Q SMA 129, F SMA 61 (60 without 2024-03-19), U SMA 63, U M1 37 and V M1 23, all within tolerance. These counts come from the first occurrences in the paper's Zenodo table, matched to the dandiset's cells (link above); the dandiset alone does not say which cells repeat, so this reading compares the paper's table with the paper's text more than it tests the dandiset. Under reading (ii), the only count the dandiset gives by itself, accepted cells summed over sessions, the counts are 160, 87, 85, 43 and 25, and only V is within tolerance.

## cells-tracked-18pct (blocked)

> About 18% of cells were tracked across up to three recording sessions.

The dandiset carries no cross-session identity (the `PlaneSegmentation` ids are 0..n-1 in every file, data check), and the Zenodo table lists each cell only at its first
occurrence: the later occurrences of a cell are not named. What can be computed is a bound. Each
session's accepted cells minus the cells first seen in it are repeat occurrences; with at most three
sessions per cell, the number of cells seen more than once is between half the repeats and all of
them.

In [ ]:
rep = 0
per_s = []
for k, d in sorted(ses.items()):
    n = (d.get("spontaneous") or d.get("task"))["events"].shape[1]
    new = ((first.subject == k[0]) & (first.region == k[1]) & (first.date == k[2])).sum()
    per_s.append((k, n, new, n - new))
    rep += n - new
U = int(uniq.sum())
print(pd.DataFrame(per_s, columns=["session", "accepted", "first seen here", "repeats"]).to_string())
print("repeat occurrences %d; unique cells %d; cells seen in 2 or more sessions between %d and %d, a fraction of %.3f to %.3f"
      % (rep, U, int(np.ceil(rep / 2)), rep, np.ceil(rep / 2) / U, rep / U))

## Figure 1D to 1F: event rate, IEI CV and amplitude

Cells are counted once, in the session where they were first identified (Methods, "Cell
identification": "We only report results for the first session in which cells were identified"),
taken from the Zenodo table. Descriptors are computed from the dandiset's events (method D). The
claims file's reading D (ii), the median of a cell over the sessions it appears in, cannot be run, as
the later occurrences are not identified (see above); only reading D (i) is run.

Per cell: rate (all cells), amplitude (cells with an event), IEI CV (cells with two or more intervals).
Spontaneous against task, a two-sided Wilcoxon signed-rank test paired by cell (scipy's default,
zero differences dropped, a standard choice), over cells with the measure defined in both parts.

In [ ]:
fc = first.drop(columns=["condition"])[["subject", "region", "date", "cell", "Global_cell_ID"]].drop_duplicates()
W = []
for _, r in fc.iterrows():
    d = ses[(r.subject, r.region, r.date)]
    rec = {"subject": r.subject, "region": r.region, "date": r.date, "cell": r.cell}
    for c in ("spontaneous", "task"):
        if c in d:
            row = D[(D.subject == r.subject) & (D.region == r.region) & (D.date == r.date) & (D.condition == c) & (D.cell == r.cell)].iloc[0]
            for meas in ("rate", "iei_cv", "amplitude"):
                rec[f"{meas}_{c}"] = row[meas]
    W.append(rec)
W = pd.DataFrame(W)
W["site"] = W.subject + " " + W.region
print("first-occurrence cells per site:", W.groupby("site").size().to_dict())
print("with a task part:", W.dropna(subset=["rate_task"]).groupby("site").size().to_dict())
summ = []
for site, g in W.groupby("site"):
    for meas in ("rate", "iei_cv", "amplitude"):
        for c in ("spontaneous", "task"):
            v = g[f"{meas}_{c}"].dropna() if f"{meas}_{c}" in g else pd.Series(dtype=float)
            summ.append({"site": site, "measure": meas, "condition": c, "n": len(v),
                         "median": v.median() if len(v) else np.nan, "min": v.min() if len(v) else np.nan, "max": v.max() if len(v) else np.nan})
print(pd.DataFrame(summ).round(4).to_string())

In [ ]:
def signrank(g, meas):
    a, b = g[f"{meas}_spontaneous"], g[f"{meas}_task"]
    ok = a.notna() & b.notna()
    a, b = a[ok], b[ok]
    if (a - b).abs().sum() == 0 or ok.sum() < 2:
        return ok.sum(), np.nan, a.median(), b.median()
    return ok.sum(), stats.wilcoxon(a, b).pvalue, a.median(), b.median()

groupings = {
    "(i) region": {"SMA": W[W.region == "SMA"], "M1": W[W.region == "M1"]},
    "(ii) site": {s: g for s, g in W.groupby("site") if g["rate_task"].notna().any()},
}
T = []
for gname, groups in groupings.items():
    for grp, g in groups.items():
        for meas in ("rate", "iei_cv", "amplitude"):
            n, p, ms, mt = signrank(g, meas)
            T.append({"grouping": gname, "group": grp, "measure": meas, "n": n, "p": p, "median_spont": ms, "median_task": mt})
T = pd.DataFrame(T)
for gname in groupings:
    sel = T.grouping == gname
    T.loc[sel, "q_family_a"] = h.bh(T.loc[sel, "p"])          # all measures of the grouping
    for meas in ("rate", "iei_cv", "amplitude"):
        s2 = sel & (T.measure == meas)
        T.loc[s2, "q_family_b"] = h.bh(T.loc[s2, "p"])        # this measure only
T["effect_r"] = np.nan
for i, r in T.iterrows():
    g = groupings[r.grouping][r.group]
    a, b = g[f"{r.measure}_spontaneous"], g[f"{r.measure}_task"]
    ok = a.notna() & b.notna()
    d = (a[ok] - b[ok])
    T.loc[i, "frac_spont_higher"] = (d > 0).mean() if ok.sum() else np.nan
print(T.drop(columns="effect_r").round(4).to_string())

### fig1d-rate-no-change

> Calcium event rates did not differ between the spontaneous and reaching task conditions in either SMA or M1.

In [ ]:
r = T[T.measure == "rate"]
print(r[["grouping", "group", "n", "p", "q_family_a", "q_family_b", "median_spont", "median_task", "frac_spont_higher"]].round(4).to_string())
print("readings with a significant difference (q < 0.05): family a", int((r.q_family_a < 0.05).sum()), "| family b", int((r.q_family_b < 0.05).sum()),
      "| uncorrected p < 0.05:", int((r.p < 0.05).sum()), "of", len(r))

**Result: reproduced.** No reading finds a difference in event rate between the spontaneous and task parts: over the six tests (SMA and M1 pooled, and Q SMA, U SMA, U M1, V M1), the smallest uncorrected p is 0.0666 (V M1) and no corrected q is below 0.05 in either family. Median rates are, for example, 0.0149 and 0.0166 events/s in pooled SMA and 0.0874 and 0.0865 in pooled M1. Only the first-session reading of a cell's value (D, i) could be run.

### fig1e-ieicv-no-change

> The IEI CV did not differ between the spontaneous and reaching task conditions in either SMA or M1.

In [ ]:
r = T[T.measure == "iei_cv"]
print(r[["grouping", "group", "n", "p", "q_family_a", "q_family_b", "median_spont", "median_task", "frac_spont_higher"]].round(4).to_string())
print("readings with a significant difference (q < 0.05): family a", int((r.q_family_a < 0.05).sum()), "| family b", int((r.q_family_b < 0.05).sum()),
      "| uncorrected p < 0.05:", int((r.p < 0.05).sum()), "of", len(r))
print("cells left out for fewer than two intervals in a part:", (W.iei_cv_spontaneous.isna() | W.iei_cv_task.isna())[W.rate_task.notna()].sum(),
      "of", W.rate_task.notna().sum())

**Result: reproduced.** No reading finds a difference in IEI CV between the parts: the smallest uncorrected p is 0.0841 (U SMA) and no corrected q is below 0.05. 78 of 252 cells with a task part are left out for fewer than two intervals in a part.

### fig1f-amplitude-u-sma-spont-higher

> In monkey U's SMA, event amplitude was significantly higher in the spontaneous state than during the reaching task.

In [ ]:
r = T[(T.measure == "amplitude") & (T.group == "U SMA")]
print(r[["grouping", "group", "n", "p", "q_family_a", "q_family_b", "median_spont", "median_task", "frac_spont_higher"]].round(4).to_string())
g = groupings["(ii) site"]["U SMA"]
ok = g.amplitude_spontaneous.notna() & g.amplitude_task.notna()
print("U SMA, by session: median spontaneous and task amplitude, and cells")
print(g[ok].groupby("date").agg(n=("cell", "size"), spont=("amplitude_spontaneous", "median"), task=("amplitude_task", "median")).round(3).to_string())

**Result: reproduced.** In U's SMA the spontaneous amplitude is higher than the task amplitude (medians 3.409 and 3.166, spontaneous higher in 72.7% of 44 cells), signed-rank p = 0.0017, q = 0.0204 with all measures in the family (a) and 0.0068 with this measure only (b); the paper gives p = 0.032. The spontaneous median is higher in each of the five U SMA sessions. The pooled SMA test is also significant here (q = 0.0198 and 0.0066), while Q's SMA alone is not (p = 0.1108); the paper reports only U. The paper's own table gives p = 0.0074 for its 42 cells with amplitudes in both parts. Without the two U SMA sessions whose events differ most from the table (2023-03-07 and 2023-03-14), the test gives p = 0.0078 over 33 cells (medians 3.272 and 3.123), so the result does not rest on them.

### fig1d-rate-m1-above-sma-u and fig1f-amplitude-m1-above-sma-u-task

Monkey U, M1 cells against SMA cells, two-sided Wilcoxon rank-sum test (Mann-Whitney U) per
condition, uncorrected and with Benjamini-Hochberg over the two conditions of the measure.

In [ ]:
U_ = W[W.subject == "U"]
R = []
for meas in ("rate", "amplitude", "iei_cv"):
    for c in ("spontaneous", "task"):
        a = U_[U_.region == "M1"][f"{meas}_{c}"].dropna()
        b = U_[U_.region == "SMA"][f"{meas}_{c}"].dropna()
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue
        auc = stats.mannwhitneyu(a, b).statistic / (len(a) * len(b))
        R.append({"measure": meas, "condition": c, "n_M1": len(a), "n_SMA": len(b), "median_M1": a.median(), "median_SMA": b.median(),
                  "p": p, "P(M1 > SMA)": auc})
R = pd.DataFrame(R)
for meas in ("rate", "amplitude", "iei_cv"):
    s = R.measure == meas
    R.loc[s, "q_two_conditions"] = h.bh(R.loc[s, "p"])
print(R.round(5).to_string())
# by session, for the rate
print(W[W.subject == "U"].groupby(["region", "date"]).agg(n=("cell", "size"), rate_spont=("rate_spontaneous", "median"), rate_task=("rate_task", "median")).round(4).to_string())

In [ ]:
# Which side a difference comes from: the same tests on the paper's own per-cell values (Zenodo table),
# for the same first-occurrence cells. Shown, not used to decide.
zt = first[first.subject == "U"][["region", "cell", "date"]].copy()
Zt = z[z.subject == "U"]
Rz = []
for meas, col in (("rate", "Event Rate"), ("amplitude", "Mean_Amplitude"), ("iei_cv", "CV")):
    for c, cz in (("spontaneous", "spontaneous"), ("task", "task")):
        a = Zt[(Zt.region == "M1") & (Zt.condition == cz)][col].dropna()
        b = Zt[(Zt.region == "SMA") & (Zt.condition == cz)][col].dropna()
        Rz.append({"measure": meas, "condition": c, "n_M1": len(a), "n_SMA": len(b), "median_M1": a.median(), "median_SMA": b.median(),
                   "p": stats.mannwhitneyu(a, b, alternative="two-sided").pvalue})
print("paper's table, U M1 against U SMA (rank-sum):")
print(pd.DataFrame(Rz).round(5).to_string())
zs = Zt[(Zt.region == "SMA")].pivot_table(index=["date", "cell"], columns="condition", values="Mean_Amplitude").dropna()
print("paper's table, U SMA amplitude spontaneous against task (signed-rank): n=%d, p=%.4f, medians %.3f and %.3f"
      % (len(zs), stats.wilcoxon(zs.spontaneous, zs.task).pvalue, zs.spontaneous.median(), zs.task.median()))

In [ ]:
# The U SMA tests without the two sessions whose events differ most from the paper's table
# (2023-03-07 and 2023-03-14, cell above), to see whether they decide the result.
drop = ["2023-03-07", "2023-03-14"]
Wx = W[~((W.subject == "U") & (W.region == "SMA") & W.date.isin(drop))]
g = Wx[Wx.site == "U SMA"]
n, p, ms, mt = signrank(g, "amplitude")
print("U SMA amplitude, spontaneous against task, without %s: n=%d, p=%.4f, medians %.3f and %.3f" % (drop, n, p, ms, mt))
Ux = Wx[Wx.subject == "U"]
for meas in ("rate", "amplitude"):
    for c in ("spontaneous", "task"):
        a, b = Ux[Ux.region == "M1"][f"{meas}_{c}"].dropna(), Ux[Ux.region == "SMA"][f"{meas}_{c}"].dropna()
        print("U M1 against U SMA without the two sessions, %s %s: n %d and %d, medians %.4f and %.4f, rank-sum p=%.2g"
              % (meas, c, len(a), len(b), a.median(), b.median(), stats.mannwhitneyu(a, b, alternative="two-sided").pvalue))


#### fig1d-rate-m1-above-sma-u

> In monkey U the event rate was higher in M1 than in SMA, in both the spontaneous and the reaching task conditions.

**Result: reproduced.** In monkey U the event rate is higher in M1 than in SMA in both conditions: spontaneous medians 0.0891 and 0.0042 events/s, task 0.1151 and 0.0089, rank-sum p below 1e-5 for both, with or without correction over the two conditions (P(M1 > SMA) 0.82 and 0.86). Without U SMA 2023-03-07 and 2023-03-14 the p-values are 1.9e-05 and 2e-06.

#### fig1f-amplitude-m1-above-sma-u-task

> In monkey U, event amplitude during the task was higher in M1 than in SMA.

**Result: not reproduced.** The task amplitude is higher in U's M1 than in its SMA (medians 3.384 and 3.193), but the rank-sum test gives p = 0.0988 uncorrected and q = 0.1976 corrected over the two conditions, so no reading reaches p < 0.05 (the paper gives p = 0.03). The paper's own per-cell table gives p = 0.0138 for the same comparison (medians 3.407 and 3.124), so the difference comes from the dandiset's event amplitudes, which differ from the table's (link above), not from the test. Most of the gap comes from two U SMA sessions, 2023-03-07 and 2023-03-14, where every dandiset amplitude is above the table's: without them the task test gives p = 0.041 (medians 3.384 and 3.133, 36 and 39 cells), below 0.05 uncorrected, while the corrected q over the two conditions is 0.082 (spontaneous p = 0.11). Leaving out sessions is not the claims file's method, so the outcome stays on all cells.

## Figure 2 and Figure S1: task-aligned responses (blocked)

Six claims (fig2-q-sma-modulated-34pct, fig2c-q-sma-direction, fig2-q-magnitude-no-difference,
figs1-u-sma-direction, figs1-u-m1-target-onset, figs1-u-m1-movement-onset) need the times of
target onset and movement onset, which the paper took from TTL pulses recorded on extra channels
(Methods, "Alignment to behavioral events"), and the raw traces. The data check shows the files hold
neither: no trial table, no TTL channel, and only the denoised trace. The Zenodo record of activity
aligned to behavioural events (doi:10.5281/zenodo.14976651) holds the paper's z-scored trial
windows, its output, not the event times, so it cannot place the trials in the dandiset's
recordings. These claims are blocked as `data_missing`.

The cell below runs the classification on the paper's z-scored windows. It tests the paper's
statistics on its own processed output, not the dandiset, and decides nothing.

In [ ]:
# A check on the paper's own processed data, which decides nothing: the Zenodo table of z-scored
# trial windows (doi:10.5281/zenodo.14976651), the output of the paper's alignment of raw traces to
# TTL events. Its columns are listed; then method Z's test on it: per cell and condition (target side),
# a signed-rank test across trials of the mean z in 0 to 0.9 s against the mean z in -1 to -0.1 s
# (the table's samples), with Benjamini-Hochberg over each chamber and alignment (main), per
# condition column (the code's FDR_correction.m), and uncorrected.
AL, al_md5 = h.read_zenodo_csv("14976651", "Event alignment-consolidated table.csv")
print("Zenodo aligned-activity table", al_md5, AL.shape)
print("columns:", list(AL.columns[:5]), "+", len(AL.columns) - 5, "z-scored samples from", AL.columns[5], "to", AL.columns[-1])
AL.columns = ["monkey", "gid", "trial", "side", "aligned"] + list(AL.columns[5:])
AL["gid"] = AL.gid.str.strip("' ")
AL["side"] = AL.side.astype(str)
AL["chamber"] = AL.gid.str.rsplit(" ", n=1).str[0]
zc = [c for c in AL.columns if c.startswith("z-scored")]
AL["pre"], AL["post"] = AL[zc[:10]].mean(axis=1), AL[zc[10:20]].mean(axis=1)
print(AL.groupby(["chamber", "aligned"]).agg(cells=("gid", "nunique"), trials=("trial", "size"), sides=("side", lambda v: sorted(v.unique()))).to_string())
F2 = []
for (ch, al), g in AL.groupby(["chamber", "aligned"]):
    R2 = []
    for (gid, side), gg in g.groupby(["gid", "side"]):
        R2.append({"gid": gid, "side": side, "trials": len(gg), "p": stats.wilcoxon(gg.post, gg.pre).pvalue})
    R2 = pd.DataFrame(R2)
    R2["q_all"] = h.bh(R2.p)
    R2["q_col"] = np.nan
    for s, idx in R2.groupby("side").groups.items():
        R2.loc[idx, "q_col"] = h.bh(R2.loc[idx, "p"])
    for col in ("q_all", "q_col", "p"):
        k = R2[R2[col] < 0.05].groupby("gid").size()
        F2.append({"chamber": ch, "aligned": al, "fdr": {"q_all": "chamber", "q_col": "per condition", "p": "none"}[col],
                   "cells": R2.gid.nunique(), "tests": len(R2), "modulated": len(k), "direction_related": int((k == 1).sum()),
                   "not_direction_related": int((k >= 2).sum())})
print(pd.DataFrame(F2).to_string())


## Figure 3: coactivation (method J)

Every session and condition, all cells. Events binned at 0.2 s, forward smoothed over 1 s; Jaccard
per pair, null from 1000 circular shifts of one cell (Methods); Z-Jaccard. A pair with a cell without
events has no Z-Jaccard and is left out, as the code does (`na.rm=TRUE` in
1_JaccardIndexAnalysisAllSessions.R). Each condition is a file of its own, so each file is binned on
its own (see the helper). The paper's code, `1_JaccardIndexAnalysisAllSessions.R`, defines the
proportion as pairs with Z-Jaccard above 1.96 plus pairs below -1.96, that is |Z| > 1.96, as the
claims file has it; the share above 1.96 alone (the code's `prop_sync`) is shown as a variant.

In [ ]:
JAC = []
rng = np.random.default_rng(20250527)
for key, d in sorted(ses.items()):
    for c, x in sorted(d.items()):
        ys = h.binned_logical(x)
        P = h.zjaccard(ys, n_shift=1000, rng=rng)
        P["dist_um"] = h.pair_distances(x["centroid"], P)
        P["subject"], P["region"], P["date"], P["condition"], P["bins"] = key[0], key[1], key[2], c, ys.shape[0]
        JAC.append(P)
JAC = pd.concat(JAC, ignore_index=True)
prop = JAC.groupby(["subject", "region", "date", "condition"]).agg(
    pairs=("z", "size"), defined=("z", lambda v: v.notna().sum()), bins=("bins", "first"),
    prop_abs=("z", lambda v: (v.dropna().abs() > 1.96).mean()),
    prop_pos=("z", lambda v: (v.dropna() > 1.96).mean()),
    prop_abs_all_pairs=("z", lambda v: (v.abs() > 1.96).sum() / len(v))).reset_index()
print(prop.round(4).to_string())
print("bins per file: spontaneous %d to %d (median %d), task %d to %d (median %d); the Methods give 1508 to 6547 (median 6012) and 2371 to 6041 (median 3700)"
      % (prop[prop.condition == "spontaneous"].bins.min(), prop[prop.condition == "spontaneous"].bins.max(), prop[prop.condition == "spontaneous"].bins.median(),
         prop[prop.condition == "task"].bins.min(), prop[prop.condition == "task"].bins.max(), prop[prop.condition == "task"].bins.median()))

In [ ]:
# The paper's Jaccard table, next to this attempt's proportions (shown, not used to decide).
Jz, j_md5 = h.read_zenodo_csv(h.ZENODO_JACCARD, "Jaccard_data_table (fig 3)_Tidy version.csv")
Jz["date"] = pd.to_datetime(Jz.iloc[:, 0], format="%d-%b-%y").dt.strftime("%Y-%m-%d")
Jz["condition"] = Jz["condition"].map({"Spontaneous": "spontaneous", "Reaching_task": "task"})
Jz = Jz.rename(columns={"structure": "region"})
pz = Jz.groupby(["subject", "region", "date", "condition"]).agg(pairs_paper=("z.jaccard", "size"),
       prop_abs_paper=("z.jaccard", lambda v: (v.dropna().abs() > 1.96).mean())).reset_index()
cmp_ = prop.merge(pz, on=["subject", "region", "date", "condition"], how="outer")
print("Zenodo Jaccard table", j_md5)
print(cmp_[["subject", "region", "date", "condition", "pairs", "pairs_paper", "prop_abs", "prop_abs_paper"]].round(4).to_string())
ok = cmp_.dropna(subset=["prop_abs", "prop_abs_paper"])
# cells per session in the paper's Jaccard table (from its pairs) against the dandiset's accepted cells
import math
cells_paper = Jz.groupby(["subject", "region", "date", "condition"]).size().apply(lambda n: int(round((1 + math.sqrt(1 + 8 * n)) / 2))).rename("cells_paper")
cells_here = pd.Series({(x["subject"], x["region"], x["date"], x["condition"]): x["events"].shape[1] for x in files}, name="cells_here")
cc = pd.concat([cells_paper, cells_here], axis=1).dropna()
print("session-conditions where the dandiset has more accepted cells than the paper's Jaccard table:")
print(cc[cc.cells_here != cc.cells_paper].astype(int).to_string())
print("sessions in both: %d; correlation of proportions %.3f; median |difference| %.4f"
      % (len(ok), np.corrcoef(ok.prop_abs, ok.prop_abs_paper)[0, 1], (ok.prop_abs - ok.prop_abs_paper).abs().median()))

### fig3-synchronized-pairs-exist

> Pairs of neurons with synchronous activity occur in both SMA and M1, in both the spontaneous and the task conditions.

In [ ]:
ex = prop.groupby(["region", "condition"]).agg(sessions=("pairs", "size"), with_sync_pairs=("prop_abs", lambda v: int((v > 0).sum())),
                                              median_prop=("prop_abs", "median"), min_prop=("prop_abs", "min"), max_prop=("prop_abs", "max"))
ex["null_rate_two_sided_1.96"] = 0.05
print(ex.round(4).to_string())
print("sessions with a proportion at or below the 0.05 of the null:")
print(prop[prop.prop_abs <= 0.05].groupby(["region", "condition"]).agg(sessions=("prop_abs", "size"), props=("prop_abs", lambda v: sorted(np.round(v, 3)))).to_string())
print("synchronized pairs in total:", int((JAC.z.abs() > 1.96).sum()), "of", int(JAC.z.notna().sum()), "defined pairs")

**Result: reproduced.** Every site and condition has sessions with synchronized pairs (|Z-Jaccard| > 1.96): M1 6 of 6 sessions in each condition, SMA 18 of 20 spontaneous sessions and 12 of 12 task sessions, 542 of 6089 defined pairs in all. The median proportions per session (SMA 0.0776 spontaneous and 0.0647 task; M1 0.1006 and 0.1802) are to be read against the 0.05 that a two-sided 1.96 threshold gives under the null, which the criterion reports and does not use. Eight SMA session-conditions are at or below 0.05 (spontaneous 0, 0, 0.038 and 0.044; task 0.026, 0.043, 0.044 and 0.045), so in SMA synchrony exists in the sense the criterion asks for, and in the task it is at about the chance rate in the median session (0.0647).

### fig3ab-synchrony-not-distance

> Synchrony between cell pairs was not related to the distance between their centroids, in SMA or M1.

Pearson (i) and Spearman (ii) correlation of Z-Jaccard with centroid distance (um), per session and
condition, two-sided p < 0.05. The example sessions of Figure 3A,B are not named: each U SMA and each
U M1 session is a reading. All sessions of all monkeys are shown, with the loess curves (span 1) for U.

In [ ]:
C = []
for k, g in JAC.dropna(subset=["z"]).groupby(["subject", "region", "date", "condition"]):
    dd, zz = g.dist_um.to_numpy(float), g.z.to_numpy(float)
    pr = stats.pearsonr(dd, zz) if len(g) > 2 else (np.nan, np.nan)
    sr = stats.spearmanr(dd, zz) if len(g) > 2 else (np.nan, np.nan)
    C.append({"subject": k[0], "region": k[1], "date": k[2], "condition": k[3], "pairs": len(g),
              "pearson_r": pr[0], "pearson_p": pr[1], "spearman_rho": sr[0], "spearman_p": sr[1],
              "dist_min": g.dist_um.min(), "dist_max": g.dist_um.max()})
C = pd.DataFrame(C)
print(C.round(4).to_string())
for test in ("pearson_p", "spearman_p"):
    sig = C[C.subject == "U"].assign(sig=lambda d: d[test] < 0.05).groupby(["region", "date"])["sig"].any()
    print(test, "- U sessions where neither condition is significant:", {r: list(sig[r][~sig[r]].index) for r in ("SMA", "M1")})
print("all sessions, significant correlations: pearson %d, spearman %d of %d" % ((C.pearson_p < 0.05).sum(), (C.spearman_p < 0.05).sum(), len(C)))

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess
Ug = JAC[(JAC.subject == "U")].dropna(subset=["z"])
keys = sorted(Ug.groupby(["region", "date", "condition"]).groups)
fig, axes = plt.subplots(3, 6, figsize=(18, 8), sharey=True)
for ax, k in zip(axes.flat, keys):
    g = Ug[(Ug.region == k[0]) & (Ug.date == k[1]) & (Ug.condition == k[2])]
    ax.scatter(g.dist_um, g.z, s=6, alpha=.5)
    lo = lowess(g.z, g.dist_um, frac=1.0)
    ax.plot(lo[:, 0], lo[:, 1], color="C1")
    ax.set_title(f"U {k[0]} {k[1]} {k[2][:5]}", fontsize=8)
    ax.set_ylim(-4, 12)
for ax in axes.flat[len(keys):]:
    ax.axis("off")
fig.supxlabel("centroid distance (um)")
fig.supylabel("Z-Jaccard")
plt.tight_layout()
plt.show()
print("U pairs plotted: %d; Z-Jaccard range %.2f to %.2f, %d points outside the axes (-4 to 12)"
      % (len(Ug), Ug.z.min(), Ug.z.max(), int(((Ug.z < -4) | (Ug.z > 12)).sum())))

**Result: reproduced.** Under both test readings there are U sessions without a significant correlation of Z-Jaccard with distance in either condition: with Pearson, 4 U SMA sessions and all 4 U M1 sessions; with Spearman, 5 U SMA and 4 U M1 sessions. Over all 44 session-conditions of all monkeys, 5 Pearson and 3 Spearman correlations reach p < 0.05 (|r| up to 0.433, Q 2022-12-01 spontaneous).

### fig3c-sync-no-condition-change

> The proportion of synchronized cell pairs did not differ between spontaneous and task conditions in
> SMA (p = 0.07) or M1 (p = 0.22).

Per region, two-sided Wilcoxon signed-rank test of spontaneous against task proportions, paired by
session, over sessions with both conditions; R's `wilcox.test` (paired) gives exact p-values for
these sample sizes without ties, which scipy's default also uses. The claims file asks for the code's
10-shift null for SMA as well (its SMA loop passes `nperm=10`).

In [ ]:
def fig3c(pr, col):
    out = []
    for reg in ("SMA", "M1"):
        w = pr[pr.region == reg].pivot_table(index=["subject", "date"], columns="condition", values=col).dropna()
        p = stats.wilcoxon(w.spontaneous, w.task).pvalue
        out.append({"region": reg, "sessions": len(w), "p": p, "median_spont": w.spontaneous.median(), "median_task": w.task.median(),
                    "median_change_task_minus_spont": (w.task - w.spontaneous).median(), "sessions_task_lower": int((w.task < w.spontaneous).sum())})
    return pd.DataFrame(out)
print("main, |Z| > 1.96 over defined pairs, 1000 shifts")
print(fig3c(prop, "prop_abs").round(4).to_string())
print("variant, Z > 1.96 only (the code's prop_sync)")
print(fig3c(prop, "prop_pos").round(4).to_string())
print("variant, |Z| > 1.96 over all pairs, undefined counted as not synchronized")
print(fig3c(prop, "prop_abs_all_pairs").round(4).to_string())
# the code's 10-shift null for SMA
rng10 = np.random.default_rng(10)
P10 = []
for key, d in sorted(ses.items()):
    if key[1] != "SMA":
        continue
    for c, x in sorted(d.items()):
        P = h.zjaccard(h.binned_logical(x), n_shift=10, rng=rng10)
        P10.append({"subject": key[0], "region": key[1], "date": key[2], "condition": c, "prop_abs": (P.z.dropna().abs() > 1.96).mean()})
P10 = pd.DataFrame(P10)
mix = pd.concat([P10, prop[prop.region == "M1"][["subject", "region", "date", "condition", "prop_abs"]]])
print("variant, SMA with a 10-shift null (M1 with 1000)")
print(fig3c(mix, "prop_abs").round(4).to_string())
print("paper's own table, |Z| > 1.96 over defined pairs")
print(fig3c(pz.rename(columns={"prop_abs_paper": "prop_abs"}), "prop_abs").round(4).to_string())

**Result: reproduced.** The proportion of synchronized pairs does not differ between the conditions: SMA p = 0.1514 over 12 sessions (median change -0.0198, task lower in 8), M1 p = 0.2188 over 6 sessions (median change +0.0509, task lower in 1). The variants agree: the positive tail only (0.2036 and 0.0938), all pairs as the denominator (0.4131 and 0.3125), and the code's 10-shift null for SMA (0.0771). The paper's own Jaccard table gives 0.0674 and 0.2188, the paper's 0.07 and 0.22; this attempt's per-session proportions correlate with the table's at 0.950 over 43 session-conditions (median absolute difference 0.0091).

## Histology (blocked)

histology-lens-placement and histology-gcamp-not-gabaergic rest on the histology images on Zenodo
(doi:10.5281/zenodo.15328390): whole-slide scans of MAP2, GFP and GABA labelling (13 files, about
8.6 GB, in `.ndpi`, `.svs` and `.vsi` formats). Judging where a lens track ends against an atlas, and
whether labelled cell bodies are double labelled, is an anatomist's reading of the images that no
computation in this notebook can make. The claims are blocked as `other`: the data are public and
readable, and no software is missing; what is missing is an anatomist's judgement.

## Summary

| Claim | Outcome |
|---|---|
| zenodo-descriptors-link | not reproduced (cells match by centroid; rates and amplitudes differ) |
| cell-counts | reproduced (through the Zenodo first occurrences) |
| cells-tracked-18pct | blocked: metadata insufficient (14.1% to 28.1% possible) |
| fig1d-rate-no-change | reproduced |
| fig1e-ieicv-no-change | reproduced |
| fig1f-amplitude-u-sma-spont-higher | reproduced |
| fig1d-rate-m1-above-sma-u | reproduced |
| fig1f-amplitude-m1-above-sma-u-task | not reproduced |
| fig2 and figs1 claims (six) | blocked: data missing |
| fig3-synchronized-pairs-exist | reproduced |
| fig3ab-synchrony-not-distance | reproduced |
| fig3c-sync-no-condition-change | reproduced |
| histology claims (two) | blocked: other (an anatomist's reading) |
| sequence claims (four) | see `sequences_martel_2025.ipynb` |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

Differences from the paper's method. Each condition is a separate file in the dandiset, so each is analysed on its own, with durations from its frames. The cells' first occurrences come from the paper's Zenodo table; the median over a cell's sessions (the claims file's reading D, ii) cannot be computed. The Jaccard null is drawn from all circular shifts at once (an FFT), with 1000 shifts drawn as the Methods say. For Figure 2 and Figure S1, a check on the paper's own z-scored windows (Zenodo) finds 20 Q SMA cells modulated under the chamber-wide FDR, against 43 in the paper, and 42 uncorrected; it decides nothing.

What was found about the dataset. The dandiset holds no TTL or trial data and no raw traces, so Figure 2 and Figure S1 cannot be tested; the trace it holds is the denoised one, without saying so. It carries no cross-session cell identity. Its event rates and amplitudes differ from the paper's per-cell table for most cells, and by large factors in two U SMA sessions, while the cells themselves match. In two U SMA sessions (2023-03-07 and 2023-03-14) the dandiset has far fewer spontaneous events than the table and a higher amplitude in every cell, and without them the U task amplitude test of M1 against SMA gives p = 0.041 uncorrected. The paper's Jaccard and Figure 4D tables count fewer cells than the dandiset has in four sessions. One U SMA session is sampled at 20 Hz where the Methods give 10 Hz; V has arm-reaching files though the paper says only Q and U did the task; the 2022-09-15 spontaneous file of Q has no session label; `indicator` and `location` are "N/A".